<a href="https://colab.research.google.com/github/MengOonLee/MachineLearning/blob/main/Courses/Statistics/StatisticalSimulations/StatisticalSimulations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

# Simulated Baseline 2025/2026 trip data based on your schema
np.random.seed(42)
n_trips = 1000

# Mocking historical baseline dataset
historical_data = pd.DataFrame({
    'Flight_Hours': np.random.exponential(scale=4.5, size=n_trips),
    'Class': np.random.choice(['Coach', 'Business', 'First'], size=n_trips, p=[0.65, 0.25, 0.10]),
    'Flight_CO2e': np.random.gamma(shape=2.0, scale=300, size=n_trips)
})

# Apply rule: Short flights (< 6 hours) have similar CO2e across classes
short_mask = historical_data['Flight_Hours'] < 6
# Normalize short flight CO2e to represent class parity if necessary
base_total_co2 = historical_data['Flight_CO2e'].sum()
target_2030_co2 = base_total_co2 * 0.70  # 30% reduction target

n_simulations = 10000
years = [2026, 2027, 2028, 2029, 2030]
success_count = 0
final_emissions = []

for _ in range(n_simulations):
    current_trips = historical_data.copy()

    # Simulate year-by-year policy effects up to 2030
    for year in years:
        # Policy 1: Eliminate a sampled % of short flights (< 6 hours)
        short_elimination_rate = np.random.uniform(0.03, 0.08) # 3-8% cut per year
        is_short = current_trips['Flight_Hours'] < 6
        drop_indices = current_trips[is_short].sample(frac=short_elimination_rate).index
        current_trips = current_trips.drop(drop_indices)

        # Policy 2: Downgrade a % of Business/First long-haul flights to Coach
        downgrade_rate = np.random.uniform(0.02, 0.05)
        long_haul_premium = (current_trips['Flight_Hours'] >= 6) & (current_trips['Class'].isin(['Business', 'First']))
        upgrade_to_coach = current_trips[long_haul_premium].sample(frac=downgrade_rate).index
        # Reduce CO2e of downgraded flights by ~35% (typical cabin class footprint difference)
        current_trips.loc[upgrade_to_coach, 'Flight_CO2e'] *= 0.65
        current_trips.loc[upgrade_to_coach, 'Class'] = 'Coach'

        # Organic travel volume growth/reduction shock
        growth_shock = np.random.normal(0.01, 0.03) # -2% to +4% net volume change
        current_trips['Flight_CO2e'] *= (1 + growth_shock)

    total_2030_co2 = current_trips['Flight_CO2e'].sum()
    final_emissions.append(total_2030_co2)

    if total_2030_co2 <= target_2030_co2:
        success_count += 1

probability_success = (success_count / n_simulations) * 100
p50_2030 = np.percentile(final_emissions, 50)

print(f"Baseline 2025 Total CO2e: {base_total_co2:,.2f} kg")
print(f"2030 Target (30% reduction): {target_2030_co2:,.2f} kg")
print(f"Monte Carlo Median 2030 CO2e: {p50_2030:,.2f} kg")
print(f"Probability of hitting the 30% reduction goal: {probability_success:.1f}%")

Baseline 2025 Total CO2e: 606,282.41 kg
2030 Target (30% reduction): 424,397.68 kg
Monte Carlo Median 2030 CO2e: 513,072.99 kg
Probability of hitting the 30% reduction goal: 0.3%


In [ ]:
import numpy as np
import pandas as pd

# Mocking historical baseline totals aggregated from your 2025-2026 dataset
base_total_co2_2026 = 500_000  # kg CO2e in 2026 baseline
short_haul_ratio = 0.45        # Proportion of emissions from flights < 6 hours
long_haul_ratio = 0.55         # Proportion of emissions from flights >= 6 hours

n_simulations = 10000
years = [2027, 2028, 2029, 2030]

cumulative_reductions = []

for i in range(n_simulations):
    # Sample policy intervention aggressiveness for this iteration
    # 1. Short-haul reduction via modal shift/rail (e.g., 20% to 50% cut in short flights)
    short_haul_cut = np.random.uniform(0.20, 0.50)

    # 2. Long-haul class downgrade effect (e.g., 10% to 30% reduction in long-haul footprint)
    class_downgrade_cut = np.random.uniform(0.10, 0.30)

    # 3. Annual baseline demand growth without controls (2% to 6%)
    demand_growth = np.random.normal(0.04, 0.015)

    # 4. Tech/SAF annual improvement (1% to 2.5%)
    tech_efficiency = np.random.uniform(0.01, 0.025)

    annual_co2 = []
    current_co2 = base_total_co2_2026

    for year in years:
        # Apply natural demand growth and tech efficiency
        growth_factor = (1 + demand_growth) * (1 - tech_efficiency)
        current_co2 *= growth_factor

        # Apply active policy levers starting in 2027
        if year >= 2027:
            # Short-haul reduction impact
            short_haul_component = current_co2 * short_haul_ratio * (1 - short_haul_cut)
            # Long-haul class optimization impact
            long_haul_component = current_co2 * long_haul_ratio * (1 - class_downgrade_cut)
            current_co2 = short_haul_component + long_haul_component

        annual_co2.append(current_co2)

    # Calculate cumulative emissions (2027-2030) + 2025/2026 baseline
    bau_cumulative = base_total_co2_2026 * 4  # Estimated BAU trajectory sum
    policy_cumulative = sum(annual_co2)

    pct_reduction = (bau_cumulative - policy_cumulative) / bau_cumulative
    cumulative_reductions.append(pct_reduction)

# Results Analysis
median_reduction = np.median(cumulative_reductions) * 100
p10_reduction = np.percentile(cumulative_reductions, 10) * 100
p90_reduction = np.percentile(cumulative_reductions, 90) * 100

print(f"Monte Carlo Simulation Results for 30% Reduction Target:")
print(f"P10 (Pessimistic Policy Effect): {p10_reduction:.1f}% reduction")
print(f"P50 (Median Policy Effect):      {median_reduction:.1f}% reduction")
print(f"P90 (Optimistic Policy Effect):  {p90_reduction:.1f}% reduction")

Monte Carlo Simulation Results for 30% Reduction Target:
P10 (Pessimistic Policy Effect): 37.8% reduction
P50 (Median Policy Effect):      49.0% reduction
P90 (Optimistic Policy Effect):  58.4% reduction
